# MotionFrame AI - Google Colab Notebook

This notebook allows you to generate a video animation from a static image and a camera motion prompt using AnimateDiff and Stable Diffusion.

In [ ]:
# @title 1. Install Dependencies
!pip install diffusers transformers accelerate

In [ ]:
# @title 2. Download Models
from diffusers import MotionAdapter, AnimateDiffPipeline

adapter = MotionAdapter.from_pretrained("guoyww/animatediff-motion-adapter-v1-5-2")
pipe = AnimateDiffPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", motion_adapter=adapter)

In [ ]:
# @title 3. Upload Image and Generate Video
import torch
from diffusers.utils import export_to_video
from PIL import Image
from google.colab import files
import io

# Upload the image
uploaded = files.upload()
image_path = list(uploaded.keys())[0]

# Get user input
prompt = input("Enter camera motion prompt (e.g., slow zoom in): ")
duration = int(input("Enter video duration in seconds (5, 10, or 20): "))

# Generate the video
pipe.to("cuda")
input_image = Image.open(image_path).convert("RGB")
input_image = input_image.resize((512, 512))

num_frames = duration * 8

output = pipe(
    prompt=prompt,
    negative_prompt="bad quality, worse quality",
    num_frames=num_frames,
    guidance_scale=7.5,
    num_inference_steps=25,
    image=input_image
)
frames = output.frames[0]

# Save and download the video
output_path = "output.mp4"
export_to_video(frames, output_path, fps=8)
files.download(output_path)